# train, eval et no_grad : quatre combinaisons

Mesurer sans mise à jour tout en choisissant le comportement des couches.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#entrainer-ou-evaluer).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Nous voulons maintenant observer un modèle sans continuer à l’ajuster. Une validation utilise des annotations réservées pour mesurer ; une inférence peut simplement produire une sortie.

Notre couche linéaire seule ne change pas de règle entre train et eval. Pour rendre cette distinction visible, nous ajoutons une expérience indépendante avec Dropout.

Dropout(p=0.5) masque certaines valeurs pendant l’entraînement et multiplie les valeurs conservées par deux. En eval, cette couche transmet les valeurs sans ce masquage.

La couche suivante additionne deux entrées avec un poids de 0,5 chacune. Pour [2,2], sans masquage, nous calculons 2×0,5+2×0,5=2.

| Mode du module | Gradients activés ? | Comportement Dropout | Graphe de la nouvelle sortie |
| --- | --- | --- | --- |
| train | Oui | Masquage aléatoire | Enregistré |
| train | Non | Masquage aléatoire | Non enregistré |
| eval | Oui | Aucun masquage | Enregistré |
| eval | Non | Aucun masquage | Non enregistré |

Aucune de ces combinaisons ne met les poids à jour seule : il faudrait encore backward puis step.

## Démonstration 1 · Tester les quatre réglages

Séparer le comportement du module et la construction d’un graphe.

### Préparer les bibliothèques

Sequential applique Dropout puis Linear. modele[1] désigne ici la seconde couche, dont nous fixons les poids à 0,5.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
modele = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(2, 1, bias=False))


### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [5]:
with torch.no_grad():
    modele[1].weight.fill_(0.5)


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [6]:
x = torch.tensor([[2., 2.]])


In [7]:
observer('x', x)

x forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[2.0, 2.0]]


### Répéter la même étape sur les données

Les deux boucles for choisissent le mode, puis l’activation des gradients. La graine est réinitialisée pour comparer le même tirage de Dropout.

train(entrainement) reçoit True ou False ; train(False) correspond à eval(). set_grad_enabled règle le suivi dans le bloc with.

Le tirage fixé conserve ici les deux entrées pendant train, donc la sortie vaut 4. En eval, elle vaut 2. Une autre graine pourrait changer les sorties train.

In [8]:
for entrainement in [True, False]:
    for gradients in [True, False]:
        torch.manual_seed(7)
        modele.train(entrainement)
        with torch.set_grad_enabled(gradients):
            prediction = modele(x)
        print("train", entrainement, "grad_actifs", gradients,
              "prediction", prediction.tolist(), "suivi", prediction.requires_grad)


train True grad_actifs True prediction [[4.0]] suivi True
train True grad_actifs False prediction [[4.0]] suivi False
train False grad_actifs True prediction [[2.0]] suivi True
train False grad_actifs False prediction [[2.0]] suivi False


### Calculer l’état suivant

Nous combinons eval et no_grad, sans backward ni step. La dernière sortie reste 2.

In [9]:
modele.eval()


Sequential(
  (0): Dropout(p=0.5, inplace=False)
  (1): Linear(in_features=2, out_features=1, bias=False)
)

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
with torch.no_grad():
    inference = modele(x)


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [11]:
print("inference", inference.tolist())

inference [[2.0]]


eval concerne les couches sensibles au mode. no_grad concerne les nouvelles opérations ; aucun des deux n’est une instruction d’apprentissage.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un modèle est en eval, mais sans no_grad. Sa nouvelle sortie possède-t-elle requires_grad=True ?

In [12]:
exo_modele = torch.nn.Linear(1,1).eval()
exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = ...  # Remplacer les points par votre réponse.
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

eval change le comportement des modules concernés ; il ne désactive pas autograd.

</details>

### Exercice 2

Dans no_grad, cette nouvelle opération conserve-t-elle un graphe de dérivées ?

In [13]:
exo_modele = torch.nn.Linear(1,1).eval()
with torch.no_grad():
    exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = ...  # Remplacer les points par votre réponse.
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Nous regardons la sortie d’une opération nouvelle, pas un tenseur créé avant le contexte.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_modele = torch.nn.Linear(1,1).eval()
exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = True
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_modele = torch.nn.Linear(1,1).eval()
with torch.no_grad():
    exo_sortie = exo_modele(torch.ones(1,1))
exo_gradient_actif = False
if exo_gradient_actif is Ellipsis or exo_gradient_actif is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_gradient_actif is exo_sortie.requires_grad
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [14]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()